# GameTheory-06k — FairBot par le théorème de Löb (compagnon natif de `ProgramGames.FairBot`)

Ce carnet est le **compagnon natif** du module [`ProgramGames.FairBot`](game_theory_lean/ProgramGames/FairBot.lean) du lake [`game_theory_lean`](game_theory_lean/README.md) — le niveau **L2** de l'équilibre en programmes de l'EPIC #15062 « Math for AI Safety ». Il câble le consommateur pédagogique qui manquait au module ([issue #20379](https://github.com/jsboige/CoursIA/issues/20379)).

Trois voisins le situent :

- [`GameTheory-06e`](Dynamics/Dynamics-04-Open-Source-Game-Theory-Python.ipynb) implémente `FairBot_toy` et le déclare lui-même : *« heuristiques ad hoc (locales au notebook) [...] **non** une reproduction »*. Ici, on exécute le **vrai** FairBot du lake.
- [`GameTheory-06f-Lean`](Dynamics/Dynamics-05-Agents-Bornes-Lean.ipynb) (niveau L1, agents à budget) se clôt sur une limite explicite : *« Le notebook ne formalise ni logique de prouvabilité ni théorème de Löb ni Gödel [...] Aucune cellule ne franchit cette limite. »* Ce carnet est le prolongement exact de cette limite — par le module conçu pour la franchir honnêtement.
- Le pont L3 ([`GLBridge`](../SymbolicAI/Lean/formal_logic_lean/FormalLogic/GLBridge.lean), lake `formal_logic_lean`) fournit le témoin noyau-vérifié du schéma de Löb dans la logique GL — **cité par référence, jamais importé** ici.

## 0. Ce que ce carnet fait — et ne fait pas

**Fait** : charger `ProgramGames.FairBot`, rendre chacun des théorèmes-phare par le noyau `lean4-wsl` (`#check` + `#print axioms` — aucune alternative `sorry`, aucun axiome au-delà des trois standard), rappeler la faille L1 (`ProgramGames.Basic`), et faire tourner un duel concret sur une **instance-témoin pédagogique** explicitement étiquetée.

**Ne fait pas** : dériver le théorème de Löb. Dans l'architecture du module, Löb est un **champ postulé de l'interface** `ModalProvability` — une hypothèse explicite que toute instance doit fournir, dont l'indépendance est mesurée dans le module (`bump_loeb_fails`) et la non-vacuité attestée par le témoin externe `FormalLogic.GLBridge.loeb_schema` (PR #15923). Le carnet respecte cette architecture : il ne définit aucune instance prétendant être « la » prouvabilité — uniquement un témoin trivial dont le rôle est pédagogique (exhiber ce que l'hypothèse de cohérence exclut).

## 1. Import du module

In [1]:
import ProgramGames.FairBot
import ProgramGames.Basic
open ProgramGames
open RepeatedGames PDAction

-- Le théorème central du module, chargé et vérifié par le noyau :
#print axioms ProgramGames.fairBot_fairBot_cooperation

import ProgramGames.FairBot
import ProgramGames.Basic
open ProgramGames
open RepeatedGames PDAction

-- Le théorème central du module, chargé et vérifié par le noyau :
#print axioms ProgramGames.fairBot_fairBot_cooperation
──────▶  'ProgramGames.fairBot_fairBot_cooperation' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 0

Raw input:
{"cmd": "import ProgramGames.FairBot\nimport ProgramGames.Basic\nopen ProgramGames\nopen RepeatedGames PDAction\n\n-- Le th\u00e9or\u00e8me central du module, charg\u00e9 et v\u00e9rifi\u00e9 par le noyau :\n#print axioms ProgramGames.fairBot_fairBot_cooperation"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "'ProgramGames.fairBot_fairBot_cooperation' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 0}

L'import passe : le lake `game_theory_lean` est construit sur cette machine, et le théorème `fairBot_fairBot_cooperation` — FairBot × FairBot = (C, C) — ne dépend que des trois axiomes standard de Lean (`propext`, `Classical.choice`, `Quot.sound`). **Aucun `sorry`, aucun axiome ajouté** : la coopération mutuelle est un théorème rendu par le noyau, pas une promesse.

## 2. L'interface `ModalProvability` — `box` + D1/D2/D3 + Löb postulé

Le module capture la prouvabilité par une **interface**, pas par une instance. Quatre champs : les trois dérivées de Hilbert-Bernays-Löb, puis le schéma de Löb lui-même comme *hypothèse explicite*.

In [2]:
#check @ModalProvability.nec        -- D1 : necessitation
#check @ModalProvability.dist       -- D2 : distribution
#check @ModalProvability.posIntros  -- D3 : introspection positive
#check @ModalProvability.loeb       -- schéma de Löb, CHAMP postulé

-- Les deux dérivées que la suite consomme :
#check @ProgramGames.box_mono      -- monotonie (D1 + D2)
#check @ProgramGames.loeb_russian  -- Löb « version russe »

#check @ModalProvability.nec        -- D1 : necessitation
──────▶  @ModalProvability.nec : ∀ {box : Prop → Prop} [self : ModalProvability box] {p : Prop}, p → box p
#check @ModalProvability.dist       -- D2 : distribution
──────▶  @ModalProvability.dist : ∀ {box : Prop → Prop} [self : ModalProvability box] {p q : Prop}, box (p → q) → box p → box q
#check @ModalProvability.posIntros  -- D3 : introspection positive
──────▶  @ModalProvability.posIntros : ∀ {box : Prop → Prop} [self : ModalProvability box] {p : Prop}, box p → box (box p)
#check @ModalProvability.loeb       -- schéma de Löb, CHAMP postulé
──────▶  @ModalProvability.loeb : ∀ {box : Prop → Prop} [self : ModalProvability box] {p : Prop}, box (box p → p) → box p

-- Les deux dérivées que la suite consomme :
#check @ProgramGames.box_mono      -- monotonie (D1 + D2)
──────▶  @box_mono : ∀ {box : Prop → Prop} [ModalProvability box] {p q : Prop}, (p → q) → box p → box q
#check @ProgramGames.loeb_russian  -- Löb « version russe »
──────▶  @loeb_russian : ∀ {box : Prop → Prop} [ModalProvability box] {p : Prop}, (box p → p) → p
--% env 1

Raw input:
{"cmd": "#check @ModalProvability.nec        -- D1 : necessitation\n#check @ModalProvability.dist       -- D2 : distribution\n#check @ModalProvability.posIntros  -- D3 : introspection positive\n#check @ModalProvability.loeb       -- sch\u00e9ma de L\u00f6b, CHAMP postul\u00e9\n\n-- Les deux d\u00e9riv\u00e9es que la suite consomme :\n#check @ProgramGames.box_mono      -- monotonie (D1 + D2)\n#check @ProgramGames.loeb_russian  -- L\u00f6b \u00ab version russe \u00bb", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "@ModalProvability.nec : ∀ {box : Prop → Prop} [self : ModalProvability box] {p : Prop}, p → box p"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@ModalProvability.dist : ∀ {box : Prop → Prop} [self : ModalProvability box] {p q : Prop}, box (p → q) → box p → box q"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@ModalProvability.posIntros : ∀ {box : Prop → Prop} [self : ModalProvability box] {p : Prop}, box p → box (box p)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@ModalProvability.loeb : ∀ {box : Prop → Prop} [self : ModalProvability box] {p : Prop}, box (box p → p) → box p"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@box_mono : ∀ {box : Prop → Prop} [ModalProvability box] {p q : Prop}, (p → q) → box p → box q"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@loeb_russian : ∀ {box : Prop → Prop} [ModalProvability box] {p : Prop}, (box p → p) → p"}],
 "env": 1}

La **version russe** est la forme directement consommée par la coopération mutuelle, et elle se lit littérairement :

$$\texttt{loeb\_russian} : (\box p \to p) \to p$$

« *Si je coopère dès que ma coopération est prouvable, alors je coopère.* » C'est le levier du point fixe : il transforme une clause réflexive en fait.

## 3. Le champ Löb n'est pas redondant — le témoin d'indépendance

Une interface à quatre champs où le quatrième dériverait des trois premiers serait une interface à trois champs déguisée. Le module le **mesure** : la modalité témoin `bump p := p ∨ (0 = 1)` satisfait D1/D2/D3 mais viole le schéma de Löb.

In [3]:
#check @ProgramGames.bump_loeb_fails
#print axioms ProgramGames.bump_loeb_fails

#check @ProgramGames.bump_loeb_fails
──────▶  bump_loeb_fails : ¬(ProgramGames.bump✝ (ProgramGames.bump✝ False → False) → ProgramGames.bump✝ False)
#print axioms ProgramGames.bump_loeb_fails
──────▶  'ProgramGames.bump_loeb_fails' does not depend on any axioms
--% env 2

Raw input:
{"cmd": "#check @ProgramGames.bump_loeb_fails\n#print axioms ProgramGames.bump_loeb_fails", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "bump_loeb_fails : ¬(ProgramGames.bump✝ (ProgramGames.bump✝ False → False) → ProgramGames.bump✝ False)"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "'ProgramGames.bump_loeb_fails' does not depend on any axioms"}],
 "env": 2}

`bump_loeb_fails` énonce `¬ (bump (bump False → False) → bump False)` : l'antécédent du schéma est vrai (réfuter `0 = 1` est une tautologie décidable), la conclusion est fausse. **Aucune preuve de Löb ne peut donc sortir des seules dérivées HBL** — le champ `loeb` a un contenu propre. Sa non-vacuité (le fait qu'une vraie instance existe) est attestée par le témoin externe noyau-vérifié `FormalLogic.GLBridge.loeb_schema`, dans le lake `formal_logic_lean` — cité par référence, le build de `game_theory_lean` n'étant volontairement pas couplé au lake GL.

## 4. FairBot : la définition

`modalAction P` coopère si `P` tient, dévie sinon — non calculable (`Prop` abstraite) : le contenu du module est dans les **théorèmes**, pas dans l'exécution. FairBot instancie `P := box r` : coopérer exactement quand la coopération **de cet adversaire** est *prouvable*.

In [4]:
#check @ProgramGames.modalAction
#check @ProgramGames.fairBotAction
#check @ProgramGames.fairBot_cooperates_with_provable_cooperator
#check @ProgramGames.fairBot_defects_without_proof

#check @ProgramGames.modalAction
──────▶  modalAction : Prop → PDAction
#check @ProgramGames.fairBotAction
──────▶  fairBotAction : (Prop → Prop) → Prop → PDAction
#check @ProgramGames.fairBot_cooperates_with_provable_cooperator
──────▶  @fairBot_cooperates_with_provable_cooperator : ∀ {box : Prop → Prop} [ModalProvability box] {r : Prop},
  box r → fairBotAction box r = cooperate
#check @ProgramGames.fairBot_defects_without_proof
──────▶  @fairBot_defects_without_proof : ∀ {box : Prop → Prop} [ModalProvability box] {r : Prop},
  ¬box r → fairBotAction box r = defect
--% env 3

Raw input:
{"cmd": "#check @ProgramGames.modalAction\n#check @ProgramGames.fairBotAction\n#check @ProgramGames.fairBot_cooperates_with_provable_cooperator\n#check @ProgramGames.fairBot_defects_without_proof", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "modalAction : Prop → PDAction"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "fairBotAction : (Prop → Prop) → Prop → PDAction"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@fairBot_cooperates_with_provable_cooperator : ∀ {box : Prop → Prop} [ModalProvability box] {r : Prop},\n  box r → fairBotAction box r = cooperate"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@fairBot_defects_without_proof : ∀ {box : Prop → Prop} [ModalProvability box] {r : Prop},\n  ¬box r → fairBotAction box r = defect"}],
 "env": 3}

De théorèmes jumeaux fixent la lecture : FairBot **coopère dès que** la coopération adverse est prouvable, et **dévie en l'absence de preuve**. La décision ne regarde ni le comportement ni la surface de l'adversaire — seulement un fait de prouvabilité.

## 5. L1 contre L2 : la faille du sondage de surface

Pourquoi changer de niveau ? Parce que le niveau borné (L1, [`ProgramGames.Basic`](game_theory_lean/ProgramGames/Basic.lean)) a une faille **prouvée** : `probeBot`, le surrogate de FairBot à budget fini, sonde le *comportement* de l'adversaire — et ce sondage a une signature de surface, lisible par un agent borné.

In [5]:
#check @ProgramGames.probeBot
#check @ProgramGames.exploiterBot
#check @ProgramGames.probeBot_exploited_by_exploiterBot

#check @ProgramGames.probeBot
──────▶  probeBot : ProgramAgent
#check @ProgramGames.exploiterBot
──────▶  exploiterBot : ProgramAgent
#check @ProgramGames.probeBot_exploited_by_exploiterBot
──────▶  probeBot_exploited_by_exploiterBot : ¬Unexploitable probeBot exploiterBot
--% env 4

Raw input:
{"cmd": "#check @ProgramGames.probeBot\n#check @ProgramGames.exploiterBot\n#check @ProgramGames.probeBot_exploited_by_exploiterBot", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "probeBot : ProgramAgent"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "exploiterBot : ProgramAgent"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "probeBot_exploited_by_exploiterBot : ¬Unexploitable probeBot exploiterBot"}],
 "env": 4}

`probeBot_exploited_by_exploiterBot` est le **constat d'échec qui motive L2** : au niveau borné, le coopérant par sondage EST exploité, théorème à l'appui. FairBot ferme cette faille — non pas en sondant mieux, mais en changeant d'objet : amener FairBot à coopérer pendant qu'on dévie exigerait `box r` avec `¬ r`, une « preuve » fausse — c'est-à-dire un système de preuve non fiable. Le saut conceptuel : **ne sonder ni le comportement ni la surface, mais la preuve**.

## 6. Le duel FairBot × FairBot : le point fixe coordonné

La structure `FairBotDuel` packe un système d'équations modales croisées : la ligne coopère *ssi* la coopération de la colonne est prouvable, et réciproquement. Le théorème central en dérive la coopération des deux côtés.

In [6]:
#check @ProgramGames.FairBotDuel
#check @ProgramGames.loeb_mutual_fixedpoint
#check @ProgramGames.fairBot_fairBot_cooperation
#print axioms ProgramGames.loeb_mutual_fixedpoint

#check @ProgramGames.FairBotDuel
──────▶  FairBotDuel : (Prop → Prop) → Type
#check @ProgramGames.loeb_mutual_fixedpoint
──────▶  @loeb_mutual_fixedpoint : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),
  d.rowCooperates ∧ d.colCooperates
#check @ProgramGames.fairBot_fairBot_cooperation
──────▶  @fairBot_fairBot_cooperation : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),
  (modalAction d.rowCooperates, modalAction d.colCooperates) = (cooperate, cooperate)
#print axioms ProgramGames.loeb_mutual_fixedpoint
──────▶  'ProgramGames.loeb_mutual_fixedpoint' does not depend on any axioms
--% env 5

Raw input:
{"cmd": "#check @ProgramGames.FairBotDuel\n#check @ProgramGames.loeb_mutual_fixedpoint\n#check @ProgramGames.fairBot_fairBot_cooperation\n#print axioms ProgramGames.loeb_mutual_fixedpoint", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "FairBotDuel : (Prop → Prop) → Type"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@loeb_mutual_fixedpoint : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),\n  d.rowCooperates ∧ d.colCooperates"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@fairBot_fairBot_cooperation : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),\n  (modalAction d.rowCooperates, modalAction d.colCooperates) = (cooperate, cooperate)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'ProgramGames.loeb_mutual_fixedpoint' does not depend on any axioms"}],
 "env": 5}

La dérivation (Barasz et al. 2014) : chaque côté satisfait la condition russe — `box q → q` — *via* sa propre clause, la monotonie `box_mono` et **D3** (une preuve de coopération devient prouvablement une preuve : c'est la sortie exacte de L1, où la coopération mutuelle ne tenait que par sondage). Löb russe conclut deux fois. Aucun `sorry` dans les axiomes.

## 7. Une instance-témoin pédagogique — explicitement étiquetée

Le module ne définit **aucune** instance de `ModalProvability` (voir son docstring) : toute instance devra fournir le champ `loeb`. Pour *faire tourner* un duel concret, ce carnet définit `toyBox` — « tout est prouvable », le **système incohérent** des manuels. Elle satisfait l'interface trivialement, et son rôle est précisément d'exhiber ce que l'hypothèse de cohérence de `fairBot_vs_defectBot` **exclut**. Ce n'est pas une modalité de prouvabilité réelle, et aucune cellule ne le prétend.

In [7]:
/-- Instance-témoin PÉDAGOGIQUE : « tout est prouvable » (système incohérent).
Satisfait trivialement D1/D2/D3 et le schéma de Löb. Elle ne représente AUCUNE
modalité de prouvabilité réelle — voir la cellule précédente. -/
instance toyBox : ModalProvability (fun (_ : Prop) => True) where
  nec _ := True.intro
  dist _ _ := True.intro
  posIntros _ := True.intro
  loeb _ := True.intro

/-- Le duel concret de la section précédente tourne : FairBot contre CoopBot. -/
def duelToy : FairBotDuel (fun _ : Prop => True) := coopDuel

theorem coopToy :
    (modalAction duelToy.rowCooperates, modalAction duelToy.colCooperates)
      = (cooperate, cooperate) :=
  fairBot_fairBot_cooperation duelToy

#print axioms coopToy

/-- Instance-témoin PÉDAGOGIQUE : « tout est prouvable » (système incohérent).
Satisfait trivialement D1/D2/D3 et le schéma de Löb. Elle ne représente AUCUNE
modalité de prouvabilité réelle — voir la cellule précédente. -/
instance toyBox : ModalProvability (fun (_ : Prop) => True) where
  nec _ := True.intro
  dist _ _ := True.intro
  posIntros _ := True.intro
  loeb _ := True.intro

/-- Le duel concret de la section précédente tourne : FairBot contre CoopBot. -/
def duelToy : FairBotDuel (fun _ : Prop => True) := coopDuel

theorem coopToy :
    (modalAction duelToy.rowCooperates, modalAction duelToy.colCooperates)
      = (cooperate, cooperate) :=
  fairBot_fairBot_cooperation duelToy

#print axioms coopToy
──────▶  'coopToy' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 6

Raw input:
{"cmd": "/-- Instance-t\u00e9moin P\u00c9DAGOGIQUE : \u00ab tout est prouvable \u00bb (syst\u00e8me incoh\u00e9rent).\nSatisfait trivialement D1/D2/D3 et le sch\u00e9ma de L\u00f6b. Elle ne repr\u00e9sente AUCUNE\nmodalit\u00e9 de prouvabilit\u00e9 r\u00e9elle \u2014 voir la cellule pr\u00e9c\u00e9dente. -/\ninstance toyBox : ModalProvability (fun (_ : Prop) => True) where\n  nec _ := True.intro\n  dist _ _ := True.intro\n  posIntros _ := True.intro\n  loeb _ := True.intro\n\n/-- Le duel concret de la section pr\u00e9c\u00e9dente tourne : FairBot contre CoopBot. -/\ndef duelToy : FairBotDuel (fun _ : Prop => True) := coopDuel\n\ntheorem coopToy :\n    (modalAction duelToy.rowCooperates, modalAction duelToy.colCooperates)\n      = (cooperate, cooperate) :=\n  fairBot_fairBot_cooperation duelToy\n\n#print axioms coopToy", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 6},
   "data":
   "'coopToy' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 6}

La coopération (C, C) est rendue sur un duel concret — et `#print axioms` montre honnêtement ce qu'elle vaut **ici** : elle repose sur le `loeb` trivial de `toyBox` (le champ de l'instance, pas un axiome). Le cercle est visible et étiqueté : c'est exactement la leçon de l'architecture du module — la solidité de la coopération est **au plus** la solidité du champ Löb de l'instance. Pour une instance réelle, ce champ est le témoin GL (L3, cité par référence).

Le revers de `toyBox`, tout aussi instructif :

In [8]:
-- Avec toyBox, « tout est prouvable » — y compris la coopération de DefectBot :
example : fairBotAction (fun _ : Prop => True) False = cooperate :=
  fairBot_cooperates_with_provable_cooperator True.intro

-- Et l'hypothèse de cohérence de fairBot_vs_defectBot devient indisponible :
-- ¬ box False = ¬ True. C'est PRÉCISÉMENT ce que cette hypothèse exclut.
#check @ProgramGames.fairBot_vs_defectBot

-- Avec toyBox, « tout est prouvable » — y compris la coopération de DefectBot :
example : fairBotAction (fun _ : Prop => True) False = cooperate :=
  fairBot_cooperates_with_provable_cooperator True.intro

-- Et l'hypothèse de cohérence de fairBot_vs_defectBot devient indisponible :
-- ¬ box False = ¬ True. C'est PRÉCISÉMENT ce que cette hypothèse exclut.
#check @ProgramGames.fairBot_vs_defectBot
──────▶  @fairBot_vs_defectBot : ∀ {box : Prop → Prop} [ModalProvability box], ¬box False → fairBotAction box False = defect
--% env 7

Raw input:
{"cmd": "-- Avec toyBox, \u00ab tout est prouvable \u00bb \u2014 y compris la coop\u00e9ration de DefectBot :\nexample : fairBotAction (fun _ : Prop => True) False = cooperate :=\n  fairBot_cooperates_with_provable_cooperator True.intro\n\n-- Et l'hypoth\u00e8se de coh\u00e9rence de fairBot_vs_defectBot devient indisponible :\n-- \u00ac box False = \u00ac True. C'est PR\u00c9CIS\u00c9MENT ce que cette hypoth\u00e8se exclut.\n#check @ProgramGames.fairBot_vs_defectBot", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@fairBot_vs_defectBot : ∀ {box : Prop → Prop} [ModalProvability box], ¬box False → fairBotAction box False = defect"}],
 "env": 7}

Un système incohérent « prouve » tout, donc FairBot coopère même avec DefectBot — et l'hypothèse `¬ box False` de `fairBot_vs_defectBot` (dévier contre DefectBot) est exactement le garde-fou : elle est fausse pour `toyBox`, par construction. L'hypothèse n'est pas cosmétique ; elle est **la ligne de séparation** entre défection mutuelle et coopération illusoire.

## 8. La table pédagogique : FairBot contre les bots standards

Trois confrontations spécialisées, plus le volet d'inexploitabilité — chacune avec son hypothèse explicite, jamais cachée.

In [9]:
#check @ProgramGames.fairBot_vs_coopBot   -- (C, C), sans aucune hypothèse
#check @ProgramGames.fairBot_vs_defectBot  -- dévie, SOUS cohérence ¬ box False
#print axioms ProgramGames.fairBot_vs_coopBot

#check @ProgramGames.fairBot_not_sucker            -- jamais dindon, sous fiabilité locale
#check @ProgramGames.fairBot_unexploitable         -- le profil (C, D) est exclu
#check @ProgramGames.opponent_cooperates_of_sound  -- la fiabilité force la coopération

#check @ProgramGames.fairBot_vs_coopBot   -- (C, C), sans aucune hypothèse
──────▶  @fairBot_vs_coopBot : ∀ {box : Prop → Prop} [ModalProvability box], fairBotAction box True = cooperate
#check @ProgramGames.fairBot_vs_defectBot  -- dévie, SOUS cohérence ¬ box False
──────▶  @fairBot_vs_defectBot : ∀ {box : Prop → Prop} [ModalProvability box], ¬box False → fairBotAction box False = defect
#print axioms ProgramGames.fairBot_vs_coopBot
──────▶  'ProgramGames.fairBot_vs_coopBot' depends on axioms: [propext, Classical.choice, Quot.sound]

#check @ProgramGames.fairBot_not_sucker            -- jamais dindon, sous fiabilité locale
──────▶  @fairBot_not_sucker : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),
  (box d.colCooperates → d.colCooperates) → ¬(d.rowCooperates ∧ ¬d.colCooperates)
#check @ProgramGames.fairBot_unexploitable         -- le profil (C, D) est exclu
──────▶  @fairBot_unexploitable : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),
  (box d.colCooperates → d.colCooperates) →
    (modalAction d.rowCooperates, modalAction d.colCooperates) ≠ (cooperate, defect)
#check @ProgramGames.opponent_cooperates_of_sound  -- la fiabilité force la coopération
──────▶  @opponent_cooperates_of_sound : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),
  (box d.colCooperates → d.colCooperates) → d.colCooperates
--% env 8

Raw input:
{"cmd": "#check @ProgramGames.fairBot_vs_coopBot   -- (C, C), sans aucune hypoth\u00e8se\n#check @ProgramGames.fairBot_vs_defectBot  -- d\u00e9vie, SOUS coh\u00e9rence \u00ac box False\n#print axioms ProgramGames.fairBot_vs_coopBot\n\n#check @ProgramGames.fairBot_not_sucker            -- jamais dindon, sous fiabilit\u00e9 locale\n#check @ProgramGames.fairBot_unexploitable         -- le profil (C, D) est exclu\n#check @ProgramGames.opponent_cooperates_of_sound  -- la fiabilit\u00e9 force la coop\u00e9ration", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "@fairBot_vs_coopBot : ∀ {box : Prop → Prop} [ModalProvability box], fairBotAction box True = cooperate"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@fairBot_vs_defectBot : ∀ {box : Prop → Prop} [ModalProvability box], ¬box False → fairBotAction box False = defect"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'ProgramGames.fairBot_vs_coopBot' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "@fairBot_not_sucker : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),\n  (box d.colCooperates → d.colCooperates) → ¬(d.rowCooperates ∧ ¬d.colCooperates)"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "@fairBot_unexploitable : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),\n  (box d.colCooperates → d.colCooperates) →\n    (modalAction d.rowCooperates, modalAction d.colCooperates) ≠ (cooperate, defect)"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@opponent_cooperates_of_sound : ∀ {box : Prop → Prop} [ModalProvability box] (d : FairBotDuel box),\n  (box d.colCooperates → d.colCooperates) → d.colCooperates"}],
 "env": 8}

| Confrontation | Résultat | Hypothèse explicite |
|---|---|---|
| FairBot × CoopBot | **(C, C)** | aucune — `box True` tient par nécessitation |
| FairBot × DefectBot | **dévie** | cohérence du système (`¬ box False`) |
| FairBot × FairBot | **(C, C)** | le champ Löb de l'instance (par Löb, pas par sondage) |
| FairBot exploité ? | **impossible** | fiabilité locale sur la clause adverse (`box r → r`) |

Et le dernier théorème va plus loin que l'inexploitabilité : sous fiabilité locale, la version russe appliquée à la clause adverse force l'adversaire à **coopérer** — la fiabilité transforme « pas exploitable » en « coopération garantie ».

## Exercice 1 — Votre propre modalité témoin

Le module mesure l'indépendance de Löb avec `bump p := p ∨ (0 = 1)` (privé). Construisez la vôtre et vérifiez qu'elle sépare les mêmes plans.

Étapes : (1) définir `slowBox p := p ∨ ((2 : ℕ) = 3)` ; (2) prouver `slowBox_nec : p → slowBox p` (D1) ; (3) prouver que `slowBox` **viole** le schéma de Löb en `p := False` — l'antécédent `slowBox (slowBox False → False)` est vrai (réfuter `2 = 3` est décidable), la conclusion `slowBox False` est fausse ; (4) conclure en une phrase : pourquoi D3 seul ne suffit jamais à retrouver Löb.

*Indice : pour (3), imiter la structure de `bump_loeb_fails` — obtenir `hb : slowBox False` de l'hypothèse, discriminer les deux cas, chacun mène à une absurdité décidable.*

In [10]:
-- TODO etudiant : definir slowBox et separer D1 du schema de Lob
-- 1. def slowBox (p : Prop) : Prop := ...
-- 2. theorem slowBox_nec ...
-- 3. theorem slowBox_loeb_fails : ¬ (slowBox (slowBox False → False) → slowBox False) := ...
-- 4. conclusion en commentaire
#eval "Exercice 1 a completer"

-- TODO etudiant : definir slowBox et separer D1 du schema de Lob
-- 1. def slowBox (p : Prop) : Prop := ...
-- 2. theorem slowBox_nec ...
-- 3. theorem slowBox_loeb_fails : ¬ (slowBox (slowBox False → False) → slowBox False) := ...
-- 4. conclusion en commentaire
#eval "Exercice 1 a completer"
─────▶  "Exercice 1 a completer"
--% env 9

Raw input:
{"cmd": "-- TODO etudiant : definir slowBox et separer D1 du schema de Lob\n-- 1. def slowBox (p : Prop) : Prop := ...\n-- 2. theorem slowBox_nec ...\n-- 3. theorem slowBox_loeb_fails : \u00ac (slowBox (slowBox False \u2192 False) \u2192 slowBox False) := ...\n-- 4. conclusion en commentaire\n#eval \"Exercice 1 a completer\"", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 1 a completer\""}],
 "env": 9}

## Exercice 2 — L'identité n'est pas une prouvabilité

La modalité « la vérité elle-même » — `box := fun p => p` — semble la plus honnête possible. Montrez qu'elle satisfait D1/D2/D3 mais **pas** le schéma de Löb, et expliquez pourquoi c'est une bonne nouvelle.

Étapes : (1) instancier les trois champs D1/D2/D3 sur `idProv : ModalProvability (fun p => p)` (chaque preuve est essentiellement l'identité) ; (2) montrer que le champ Löb exigerait, pour `p := False`, une preuve de `(p → p) → p` — c'est-à-dire de `True → False` ; (3) conclure : une modalité de prouvabilité doit être *plus riche* que la vérité nue, faute de quoi le théorème de Löb — et la coopération FairBot × FairBot — s'effondre.

*Indice : pour (2), ne cherchez pas à prouver l'impossible — montrez que l'antécédent `box (box False → False)` se réduit à `True` pendant que la conclusion `box False` se réduit à `False`.*

In [11]:
-- TODO etudiant : l'identite satisfait D1/D2/D3 mais pas Lob
-- 1. instance idProv : ModalProvability (fun p => p) := ...
-- 2. prouver que le champ loeb serait (True → False) en p := False : impossible
-- 3. conclusion en commentaire
#eval "Exercice 2 a completer"

-- TODO etudiant : l'identite satisfait D1/D2/D3 mais pas Lob
-- 1. instance idProv : ModalProvability (fun p => p) := ...
-- 2. prouver que le champ loeb serait (True → False) en p := False : impossible
-- 3. conclusion en commentaire
#eval "Exercice 2 a completer"
─────▶  "Exercice 2 a completer"
--% env 10

Raw input:
{"cmd": "-- TODO etudiant : l'identite satisfait D1/D2/D3 mais pas Lob\n-- 1. instance idProv : ModalProvability (fun p => p) := ...\n-- 2. prouver que le champ loeb serait (True \u2192 False) en p := False : impossible\n-- 3. conclusion en commentaire\n#eval \"Exercice 2 a completer\"", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "\"Exercice 2 a completer\""}],
 "env": 10}

## Exercice 3 — Le duel asymétrique contre DefectBot

Construisez le duel de FairBot contre un adversaire qui ne coopère **jamais**, et lisez ce que les équations imposent.

Étapes : (1) définir `defectDuel (box) [ModalProvability box] : FairBotDuel box` avec `colCooperates := False` et les clauses allant avec (`rowCooperates` doit être équivalent à `box False` pour la ligne ; la colonne, qui ne coopère jamais, a une clause triviale à écrire soigneusement) ; (2) prouver `fairBot_vs_defectBot` *via* votre duel : sous cohérence `¬ box False`, la ligne dévie ; (3) vérifier sur `toyBox` que sans cohérence, le même duel donne la coopération illusoire de la section 7.

*Indice : pour la clause de colonne, `False ↔ box rowCooperates` demande une preuve dans le sens `box rowCooperates → False` — laquelle n'existe pas en général ! Réfléchissez : quel `rowCooperates` rend les deux clauses satisfaisables ? (Ce n'est pas `box False` seul : la clause de ligne le force, et alors `box (box False) → False`... à vous de voir pourquoi la cohérence ferme la boucle.)*

In [12]:
-- TODO etudiant : duel FairBot x DefectBot
-- 1. def defectDuel ... : FairBotDuel box := ...
-- 2. sous coherence, la ligne devie (adapter fairBot_vs_defectBot)
-- 3. sur toyBox : la meme structure donne la cooperation illusoire
#eval "Exercice 3 a completer"

-- TODO etudiant : duel FairBot x DefectBot
-- 1. def defectDuel ... : FairBotDuel box := ...
-- 2. sous coherence, la ligne devie (adapter fairBot_vs_defectBot)
-- 3. sur toyBox : la meme structure donne la cooperation illusoire
#eval "Exercice 3 a completer"
─────▶  "Exercice 3 a completer"
--% env 11

Raw input:
{"cmd": "-- TODO etudiant : duel FairBot x DefectBot\n-- 1. def defectDuel ... : FairBotDuel box := ...\n-- 2. sous coherence, la ligne devie (adapter fairBot_vs_defectBot)\n-- 3. sur toyBox : la meme structure donne la cooperation illusoire\n#eval \"Exercice 3 a completer\"", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "\"Exercice 3 a completer\""}],
 "env": 11}

## Conclusion — ce que ce carnet a rendu visible

| Résultat rendu par le noyau | Ce qu'il dit |
|---|---|
| `fairBot_fairBot_cooperation` | FairBot × FairBot = (C, C), **par Löb** — là où L1 n'avait que le sondage |
| `bump_loeb_fails` | le champ Löb de l'interface a un contenu propre (indépendance mesurée) |
| `probeBot_exploited_by_exploiterBot` | la faille L1 qui motive le saut de niveau |
| `fairBot_not_sucker` / `fairBot_unexploitable` | jamais dindon — sous hypothèse de fiabilité explicite |
| `fairBot_vs_coopBot` / `fairBot_vs_defectBot` | la table pédagogique, hypothèses apparentes |
| `coopToy` (instance-témoin) | la solidité de la coopération est celle du champ Löb de l'instance |

**La limite honnête, répétée** : Löb est postulé au champ de l'interface — ce carnet ne le dérive pas, et son instance-témoin ne prétend pas être une prouvabilité réelle. La chaîne de confiance traverse trois livrables : l'indépendance mesurée ici (`bump_loeb_fails`), la non-vacuité attestée par le témoin GL noyau-vérifié (L3, lake `formal_logic_lean`, PR #15923), et la dérivation de Barasz et al. (2014) formalisée dans le module.

**Ponts** : les `_toy` de [06e](Dynamics/Dynamics-04-Open-Source-Game-Theory-Python.ipynb) (heuristiques déclarées), la limite explicite de [06f-Lean](Dynamics/Dynamics-05-Agents-Bornes-Lean.ipynb) (L1 s'arrête avant Löb — ce carnet la franchit), le pont GL de [Lean-34b](../SymbolicAI/Lean/Lean-34b-FairBot-Loeb.ipynb) (L3, logique de la prouvabilité).

## Sources

- Barasz, Christiano, Fallenstein, Herreshoff, LaVictore & Yudkowsky (2014), « Robust Cooperation in the Prisoner's Dilemma: Program Equilibrium via Succinct Circuit Representation », arXiv:1401.5577.
- LaVictor (2015), « Robust cooperation in the Prisoner's Dilemma: program equilibrium via provability logic », arXiv:1401.5577 appendices.
- Critch (2016), « Parametric Bounded Löb's Theorem and Robust Cooperation of Bounded Agents », arXiv:1602.04184.

Carnet livré par l'issue #20379 — le module `ProgramGames.FairBot` a désormais son consommateur pédagogique.